Salary parsing: pay_period x amounts -> annual USD (Day 8).
Reads:  jobmarket.silver.stg_postings_kaggle
Writes: same table, rebuilt with salary_min/salary_max (annual USD)
Method: distribution check -> tested conversion function -> sanity bounds.
Keeps _raw columns for auditability.

In [0]:
from pyspark.sql import functions as F

k = spark.table("jobmarket.silver.stg_postings_kaggle")

# which periods exist, how often?
k.groupBy("pay_period").count().orderBy(F.desc("count")).display()

# value ranges per period — do the magnitudes make sense?
(k.filter(F.col("salary_min_raw").isNotNull())
  .groupBy("pay_period")
  .agg(F.count("*").alias("n"),
       F.min("salary_min_raw").alias("min_of_min"),
       F.expr("percentile_approx(salary_min_raw, 0.5)").alias("median_min"),
       F.max("salary_max_raw").alias("max_of_max"))
  .display())

In [0]:
def annualize_salary(amount_col, period_col):
    """Amount + period -> annual USD in [10k, 1M], else null. (Docstring as before +)
       Post-conversion band: this platform defines 'salary' as $10k-1M
       annual full-time equivalent; outside -> null."""
    annual = (
        F.when((period_col == "YEARLY")   & (amount_col >= 5000),               amount_col)
         .when((period_col == "MONTHLY")  & (amount_col.between(500, 50000)),   amount_col * 12)
         .when((period_col == "BIWEEKLY") & (amount_col.between(200, 20000)),   amount_col * 26)
         .when((period_col == "WEEKLY")   & (amount_col.between(100, 10000)),   amount_col * 52)
         .when((period_col == "HOURLY")   & (amount_col.between(5, 500)),       amount_col * 2080)
         .when(period_col.isNull() & (amount_col.between(5, 200)),              amount_col * 2080)
         .when(period_col.isNull() & (amount_col.between(20000, 1000000)),      amount_col)
         .otherwise(F.lit(None))
    )
    return F.when(annual.between(10000, 1000000), annual).otherwise(F.lit(None))

In [0]:
# ---- Unit tests: (amount, period, expected_annual) ----
# Permanent regression guard: any future change that breaks a conversion
# turns this cell red instead of silently poisoning Gold.

test_data = [
    # happy paths: labeled periods within gates
    (17.0,     "HOURLY",   35360.0),
    (25.0,     "HOURLY",   52000.0),
    (5000.0,   "MONTHLY",  60000.0),
    (95000.0,  "YEARLY",   95000.0),
    (1500.0,   "WEEKLY",   78000.0),
    (2057.0,   "BIWEEKLY", 53482.0),

    # dirty labels: values failing their period's gate -> null
    (275000.0, "HOURLY",   None),      # the impostor from Cell 1
    (1.0,      "YEARLY",   None),
    (1.0,      "MONTHLY",  None),
    (600.0,    "HOURLY",   None),

    # null period: magnitude inference
    (25.0,     None,       52000.0),
    (95000.0,  None,       95000.0),
    (5000.0,   None,       None),      # ambiguous middle -> honest null
    (250.0,    None,       None),
    (2000000.0, None,      None),

    (85000000.0, "YEARLY", None),   # fat-finger yearly, above 1M band
    (8000.0,     "YEARLY", None),   # legit label, below 10k band

    # null amounts stay null
    (None,     "YEARLY",   None),
    (None,     None,       None),
]

from pyspark.sql.types import StructType, StructField, DoubleType, StringType
schema = StructType([
    StructField("amount",   DoubleType(), True),
    StructField("period",   StringType(), True),
    StructField("expected", DoubleType(), True),
])
test_df = spark.createDataFrame(test_data, schema)

result = test_df.withColumn("got", annualize_salary(F.col("amount"), F.col("period")))

# <=> = null-safe equality (plain == returns null for null cases and
# would silently skip exactly the tests that check null behavior)
failures = result.filter(~F.col("got").eqNullSafe(F.col("expected")))
n_fail = failures.count()

if n_fail > 0:
    print(f"❌ {n_fail} test case(s) FAILED:")
    failures.display()
assert n_fail == 0, f"{n_fail} salary test case(s) failed"

print(f"✅ All {result.count()} salary conversion tests passed")
display(result)

In [0]:
k = spark.table("jobmarket.silver.stg_postings_kaggle")

k2 = (k
    .withColumn("salary_min_annual", annualize_salary(F.col("salary_min_raw"), F.col("pay_period")))
    .withColumn("salary_max_annual", annualize_salary(F.col("salary_max_raw"), F.col("pay_period")))
)


# min > max -> null both (can't know which side is wrong; refuse to guess)
k2 = (k2
    .withColumn("salary_min",
        F.when(F.col("salary_min_annual") > F.col("salary_max_annual"), F.lit(None))
         .otherwise(F.col("salary_min_annual")))
    .withColumn("salary_max",
        F.when(F.col("salary_min_annual") > F.col("salary_max_annual"), F.lit(None))
         .otherwise(F.col("salary_max_annual")))
    .drop("salary_min_annual", "salary_max_annual")
)

display(k2.select("salary_min_raw", "salary_max_raw", "pay_period",
                  "salary_min", "salary_max")
         .filter(F.col("salary_min_raw").isNotNull()).limit(15))

In [0]:
k.filter(F.col("pay_period").isNull() & F.col("salary_min_raw").isNotNull()).count()

In [0]:
(k2.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("jobmarket.silver.stg_postings_kaggle"))

In [0]:
s = spark.table("jobmarket.silver.stg_postings_kaggle")

# (a) does the annualized distribution look like real salaries?
s.filter(F.col("salary_min").isNotNull()).select(
    F.count("*").alias("rows_with_salary"),
    F.round(F.expr("percentile_approx(salary_min, 0.25)")).alias("p25"),
    F.round(F.expr("percentile_approx(salary_min, 0.5)")).alias("median"),
    F.round(F.expr("percentile_approx(salary_min, 0.75)")).alias("p75"),
    F.round(F.min("salary_min")).alias("min"),
    F.round(F.max("salary_min")).alias("max"),
).display()

# (b) how many raw values did the gates reject?
had_raw    = s.filter(F.col("salary_min_raw").isNotNull()).count()
got_annual = s.filter(F.col("salary_min").isNotNull()).count()
print(f"Raw salary values: {had_raw}")
print(f"Converted: {got_annual}")
print(f"Rejected by gates/inference: {had_raw - got_annual} "
      f"({(had_raw - got_annual)/had_raw*100:.1f}%)")

# (c) cross-check vs the dataset's own normalized_salary
comp = (spark.table("jobmarket.bronze.job_postings_kaggle")
    .select("job_id", F.col("normalized_salary").cast("double").alias("their_norm"))
    .join(s.select(F.col("source_posting_id").alias("job_id"),
                   "salary_min", "salary_max"), "job_id")
    .filter(F.col("their_norm").isNotNull() & F.col("salary_min").isNotNull())
    .withColumn("ours_mid", (F.col("salary_min") + F.col("salary_max")) / 2)
    .withColumn("ratio", F.col("ours_mid") / F.col("their_norm")))

comp.select(
    F.count("*").alias("comparable_rows"),
    F.round(F.expr("percentile_approx(ratio, 0.5)"), 3).alias("median_ratio"),
    F.round(F.avg((F.col("ratio").between(0.9, 1.1)).cast("int")) * 100, 1).alias("pct_within_10pct"),
).display()